In [1]:
import sys
import os
from pathlib import Path
# Ensure the repository root is on sys.path so imports work when running the notebook
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Added to sys.path:', repo_root)

Added to sys.path: C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai


In [2]:
# Import the extractor (adjusts to package structure in the workspace)
try:
    from medical_potential.moa_innovation import get_moa_innovation_assessment
except Exception as e:
    print('Import failed:', e)
    raise

c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\.medical-potential-venv\Lib\site-packages\google\auth\transport\grpc.py:43: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(


In [3]:
result = await get_moa_innovation_assessment(molecule_name='Mazdutide')

[INFO] [DIMENSION START] Running dimension 'MoA Innovation' for molecule 'Mazdutide'
[INFO] ================================================================================
[INFO] MOA INNOVATION ASSESSMENT
[INFO] Drug: Mazdutide
[INFO] Started: 2026-11-05 20:07:41
[INFO] ================================================================================
[INFO] [MoA] Step 1/13 - Retrieving MoA data from BigQuery
[INFO] [MoA Step 1] Querying BigQuery for Mazdutide MoA data...
[INFO]   Found 1 indication(s): ['Overweight, Obesity, Type 2 Diabetes']
[INFO]   Using: MoA=Glucagon Like Peptide 1 Receptor Agonist; Glucagon Receptor Agonist, Indication=Overweight, Obesity, Type 2 Diabetes
[INFO] [MoA] Using: MoA='Glucagon Like Peptide 1 Receptor Agonist; Glucagon Receptor Agonist', Indication='Overweight, Obesity, Type 2 Diabetes'
[INFO] [MoA] Step 2/13 - Building mechanism landscape
[INFO] [MoA Step 2] Building mechanism landscape for Overweight, Obesity, Type 2 Diabetes...
[INFO] AFC is enabled 


Gemini response served via mode: ON_DEMAND_FLEX


[INFO]   Found 13 approved drugs, 9 pipeline drugs
[INFO] [MoA] Step 3/13 - Classifying MoA position
[INFO] [MoA Step 3] Classifying MoA position...
[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via mode: ON_DEMAND_FLEX


[INFO]   Classification: Best-in-Class
[INFO] [MoA] Running Steps 4, 5, 6 in parallel...
[INFO] [MoA Step 4] Checking biological rationale...
[INFO] [MoA Step 5] Checking clinical validation...
[INFO] [MoA Step 6] Checking for mechanism failures...
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"
[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"



Gemini response served via mode: ON_DEMAND_FLEX


[INFO] HTTP Request: POST https://aiplatform.googleapis.com/v1/projects/cognito-dev-380506/locations/global/publishers/google/models/gemini-3-flash-preview:generateContent "HTTP/1.1 200 OK"
[INFO] Resolving 9 unique URLs with 6 workers (rate cap 8.0 req/s)...
[INFO] Resolving 2 unique URLs with 6 workers (rate cap 8.0 req/s)...



Gemini response served via mode: ON_DEMAND_FLEX

Gemini response served via mode: ON_DEMAND_FLEX


[WARNING] Retrying (Retry(total=0, connect=0, read=0, redirect=None, status=1)) after connection broken by 'SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)'))': /journals/endocrinology/articles/10.3389/fendo.2026.1933913/full
[INFO] Resolving 1 unique URLs with 6 workers (rate cap 8.0 req/s)...
[INFO] Resolved 9 / 9 URLs (100%) in 61.3s
[INFO]   Rationale: Strong
[INFO] Resolved 1 / 1 URLs (100%) in 74.6s
[INFO]   Clinically validated: True


CancelledError: 

[INFO] Resolved 2 / 2 URLs (100%) in 145.7s


In [ ]:
import pandas as pd

moa_output_data = pd.read_csv(r"C:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\notebooks\input\moa_innovation\moa_innovation_output.csv")

In [ ]:
import json

parsed_grounding_urls = moa_output_data["grounding_urls_by_step"].apply(
    lambda value: json.loads(value) if isinstance(value, str) else value
)
grounding_urls_by_step_df = pd.concat(
    [
        moa_output_data[["drug_name"]].reset_index(drop=True),
        pd.json_normalize(parsed_grounding_urls).reset_index(drop=True),
    ],
    axis=1,
)
step_columns = grounding_urls_by_step_df.columns[1:]
for step_column in step_columns:
    grounding_urls_by_step_df[f"{step_column}_length"] = grounding_urls_by_step_df[
        step_column
    ].apply(lambda urls: len(urls) if isinstance(urls, list) else 0)
ordered_columns = ["drug_name"]
for step_column in step_columns:
    ordered_columns.extend([step_column, f"{step_column}_length"])
grounding_urls_by_step_df = grounding_urls_by_step_df[ordered_columns]
grounding_urls_by_step_df